# Test single-stream memorization of random vectors
This is a unit test of the sparse memory.

In [ ]:
import random
import statistics

import torch
import torch.nn.functional as F

from model.sparse.sparse_dense_model import (
    SparseActivationDenseModelConfig,
)
from model.sparse.sparse_distributed_model import SparseDistributedModel
from util.optimizer import ModelOptimizer, ModelOptimizerConfig

In [ ]:
B = 8
I = 128  # input size
V = 4  # num values memorized
M = 2  # num value models - dense models only

C = 400  # capacity 
K = 16
E = 2  # num ensembles - boosts orthogonality and therefore capacity

NUM_DATA = 100
NUM_STEPS = 800
SINGLE_STREAM = True

In [ ]:
model_config = SparseActivationDenseModelConfig(
    hidden_size_factor = 1,
    weight_decay_factor = 0,
    name = "Sparse",
    nonlinearity = "leaky-relu",
    input_layer_norm = True,
    input_dropout = 0,
    input_weight_clip = 0.0,
    input_size = I,
    hidden_size = C * 1,
    output_size = V,
    output_nonlinearity = None,
    layers = 2,
    hidden_dropout = 0,
    bias = True,
)

m = SparseDistributedModel(
    model_configs = [model_config],
    input_key_size = I,
    input_value_size = I,
    memory_size = C,
    ensemble_size = E,
    sparsity = K,
)

optimizer = ModelOptimizer(
    config = ModelOptimizerConfig(
        name = "optimizer",
        optimizer_type = ModelOptimizer.OPTIMIZER_SGD,
        learning_rate = 0.1,
        momentum = 0.1,
        weight_decay = 0,
        clip_grad_norm = 0,
    ),
    parameters = m.get_trainable_parameters(),
)


In [ ]:
# Generate patterns to memorize
data = []
for i in range(NUM_DATA):
    k = torch.randn(1, I)
    v = torch.randn(1, V)
    data.append(
        {
            "k": k, 
            "v": v,
        }
    )    


In [ ]:
def get_minibatch(data, B:int):
    """
    data: List of dicts, e.g., [{'k': tensor, 'v': tensor}, ...]
    B: batch size
    """
    # 1. Select a random subset of size B
    subset = random.sample(data, B)

    # 2. Extract and stack into [B, K] and [B, V]
    bk = torch.stack([item['k'] for item in subset]).squeeze(1)
    bv = torch.stack([item['v'] for item in subset]).squeeze(1)
    return bk, bv

In [ ]:
for i in range(NUM_STEPS):
    sum_error = 0

    def set_value(k, v):
        p, _ = m.do_model(
            key_input = k, 
            model_input = k, 
        )
        loss = F.mse_loss(p, v)
        parameters = m.get_trainable_parameters()
        optimizer.optimize(
            loss = loss,
            parameters = parameters,
        )
        error = torch.abs(p - v).sum().item()
        return error

    if SINGLE_STREAM:
        for j in range(NUM_DATA):
            k = data[j]["k"]
            v = data[j]["v"]
            error = set_value(k, v)            
            sum_error += error

    else:
        num_batches = NUM_DATA // B
        for j in range(num_batches):
            k, v = get_minibatch(data, B)
            error = set_value(k, v)            
            sum_error += error

    if (i < 10) or ((i%50) == 0):
        print(f"Step: {i} Sum error:{sum_error}")


In [ ]:
# Measure error of stored values
errors = []
for i in range(NUM_DATA):
    k = data[i]["k"]
    v = data[i]["v"]

    p, _ = m.do_model(
        key_input = k, 
        model_input = k, 
    )

    #print(f"v:{v} p:{p}")
    e = (v-p).abs().sum().item()
    errors.append(round(e, 5))
print(f"Errors: {errors}")
mean_error = statistics.mean(errors)
print(f"Mean. Sum Abs. Error:{mean_error}")
assert(mean_error < 0.001)